# Image Moderation — TF NSFW classifier (replaces NudeNet baseline)

Trains a binary NSFW classifier on the local corpus
(`data/nsfw/out/{train,val,test}/{Neutral,NSFW}`), plus the Reddit
title->is_nsfw table as a text prior. Serves `app/moderation_engine.py`
`/api/v1/moderation/image` (currently NudeNet + pixel-analysis fallback); this
model is the trained replacement. Exported ONNX takes a 224x224x3 float image
(pre-/255) and outputs p(NSFW).

In [1]:
import importlib.util
import os
import pathlib
import sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
sys.path.insert(0, str(ai))         # for the training.* namespace package
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}

# -- shared bootstrap: seeds RNGs, caps CPU threads, resolves data/output roots ----
try:
    from training.bootstrap import *  # noqa: F403
    CFG = init()
except Exception as _boot_err:  # bootstrap is an upgrade, never brick a notebook
    print('[bootstrap] unavailable:', repr(_boot_err))
    CFG = {}
SCALE = CFG.get('scale', os.environ.get('BUDDY_SCALE', 'demo'))   # smoke | demo | full
from tf_utils import on_gpu, tf_version
from tf_utils import set_memory_growth
set_memory_growth()
print('TF', tf_version(), '| GPU:', on_gpu(), '| scale:', SCALE)


2026-08-04 23:19:15.951084: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-04 23:19:16.031303: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-04 23:19:18.035976: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TF 2.20.0 | GPU: False | scale: demo


2026-08-04 23:19:20.465597: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
# Local NSFW image corpus (class dirs: Neutral=0, NSFW=1)
import os
import tensorflow as tf
from buddy_data import nsfw_images, reddit_nsfw

root = nsfw_images()
print('root:', root)
IMG = 224
train_ds = tf.keras.utils.image_dataset_from_directory(
    root / 'train', image_size=(IMG, IMG), batch_size=32, shuffle=True)
val_ds = tf.keras.utils.image_dataset_from_directory(
    root / 'val', image_size=(IMG, IMG), batch_size=32)
test_ds = tf.keras.utils.image_dataset_from_directory(
    root / 'test', image_size=(IMG, IMG), batch_size=32)

print('class_names:', train_ds.class_names)
nsfw_df = reddit_nsfw()
print('reddit title prior:', nsfw_df.shape, nsfw_df['is_nsfw'].value_counts().to_dict())

root: /home/peter/Desktop/Buddy-Up/backend/ai_service/data/nsfw/out
Found 16638 files belonging to 2 classes.
Found 5546 files belonging to 2 classes.
Found 5547 files belonging to 2 classes.
class_names: ['NSFW', 'Neutral']
reddit title prior: (617952, 3) {False: 517475, True: 100477}


In [3]:
# Balanced class weights + augmentation (NSFW is the minority class)
import numpy as np

def count_by_class(ds):
    counts = np.zeros(len(train_ds.class_names), dtype=int)
    for _, y in ds:
        counts += np.bincount(y.numpy(), minlength=len(counts))
    return counts

tr_counts = count_by_class(train_ds)
total = tr_counts.sum()
class_weight = {i: total / (len(tr_counts) * c) for i, c in enumerate(tr_counts)}
print('train counts:', dict(zip(train_ds.class_names, tr_counts.tolist())), '| weights:', class_weight)

# Bounded prefetch buffer: AUTOTUNE over-allocates on low-RAM CPU boxes and
# caused severe slow-downs; 2 batches keeps a stable ~40 MB in flight.
AUT = tf.data.AUTOTUNE
PREFETCH = 2
aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
])
train_ds = (train_ds
            .map(lambda x, y: (x / 255.0, y), num_parallel_calls=AUT)
            .map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUT)
            .prefetch(PREFETCH))
val_ds = val_ds.map(lambda x, y: (x / 255.0, y)).prefetch(PREFETCH)
test_ds = test_ds.map(lambda x, y: (x / 255.0, y)).prefetch(PREFETCH)

if SCALE == 'smoke':                      # fast CI/verification subset
    train_ds = train_ds.take(6)
    val_ds = val_ds.take(3)
    test_ds = test_ds.take(3)


2026-08-04 23:19:49.086201: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


train counts: {'NSFW': 10051, 'Neutral': 6587} | weights: {0: np.float64(0.8276788379265745), 1: np.float64(1.2629421587976317)}


In [4]:
# Frozen MobileNetV3Small head -> staged fine-tune
base = tf.keras.applications.MobileNetV3Small(
    weights='imagenet', include_top=False, input_shape=(IMG, IMG, 3))
base.trainable = False
x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
m = tf.keras.Model(base.input, out)
m.compile(tf.keras.optimizers.Adam(1e-3), 'binary_crossentropy',
          metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
m.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv (Conv2D)       │ (None, 112, 112,  │        432 │ rescaling[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_bn             │ (None, 112, 112,  │         64 │ conv[0][0]        │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 112, 112,  │          0 │ conv_bn[0][0]     │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 113, 113,  │          0 │ activation[0][0]  │
│ (ZeroPadding2D)     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 56, 56,    │        144 │ expanded_conv_de… │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 56, 56,    │         64 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 56, 56,    │          0 │ expanded_conv_de… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 16)  │          0 │ re_lu[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 8)   │        136 │ expanded_conv_sq… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 8)   │          0 │ expanded_conv_sq… │
│ (ReLU)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 1, 1, 16)  │        144 │ expanded_conv_sq… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1, 1, 16)  │          0 │ expanded_conv_sq… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 1, 1, 16)  │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 1, 1, 16)  │          0 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_sque… │ (None, 56, 56,    │          0 │ re_lu[0][0],      │
│ (Multiply)          │ 16)               │            │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 56, 56,    │        256 │ expanded_conv_sq

 Total params: 939,697 (3.58 MB)

 Trainable params: 577 (2.25 KB)

 Non-trainable params: 939,120 (3.58 MB)

In [5]:
# Train head (frozen backbone), then fine-tune the top block (early-stopped)
# EarlyStopping + ReduceLROnPlateau make the run robust across BUDDY_SCALE:
# more epochs no longer mean overfitting, the best val_loss weights are kept.
import time
EPOCHS = {'smoke': 2, 'demo': 8, 'full': 12}[SCALE]
FT_EPOCHS = {'smoke': 0, 'demo': 3, 'full': 5}[SCALE]
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2,
                                     restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                         patience=1, min_lr=1e-6),
]
t0 = time.time()
m.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
      class_weight=class_weight, callbacks=callbacks, verbose=1)

if FT_EPOCHS > 0:
    base.trainable = True
    for layer in base.layers[: int(0.7 * len(base.layers))]:
        layer.trainable = False
    m.compile(tf.keras.optimizers.Adam(1e-5), 'binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    m.fit(train_ds, validation_data=val_ds, epochs=FT_EPOCHS,
          callbacks=callbacks, verbose=1)
print(f'total {time.time()-t0:.0f}s')


Epoch 1/8


/home/peter/Desktop/ml-env/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


520/520 ━━━━━━━━━━━━━━━━━━━━ 316s 590ms/step - accuracy: 0.5073 - auc: 0.5054 - loss: 0.6987 - val_accuracy: 0.3960 - val_auc: 0.5511 - val_loss: 0.7152 - learning_rate: 0.0010
Epoch 2/8
520/520 ━━━━━━━━━━━━━━━━━━━━ 317s 576ms/step - accuracy: 0.5027 - auc: 0.5025 - loss: 0.6966 - val_accuracy: 0.3970 - val_auc: 0.6122 - val_loss: 0.7103 - learning_rate: 0.0010
Epoch 3/8
520/520 ━━━━━━━━━━━━━━━━━━━━ 348s 667ms/step - accuracy: 0.5084 - auc: 0.5154 - loss: 0.6941 - val_accuracy: 0.4592 - val_auc: 0.6185 - val_loss: 0.6941 - learning_rate: 0.0010
Epoch 4/8
520/520 ━━━━━━━━━━━━━━━━━━━━ 303s 579ms/step - accuracy: 0.5103 - auc: 0.5128 - loss: 0.6944 - val_accuracy: 0.4439 - val_auc: 0.6280 - val_loss: 0.6963 - learning_rate: 0.0010
Epoch 5/8
520/520 ━━━━━━━━━━━━━━━━━━━━ 307s 589ms/step - accuracy: 0.5176 - auc: 0.5254 - loss: 0.6922 - val_accuracy: 0.5613 - val_auc: 0.6303 - val_loss: 0.6864 - learning_rate: 5.0000e-04
Epoch 6/8
520/520 ━━━━━━━━━━━━━━━━━━━━ 341s 654ms/step - accuracy: 0.51

In [6]:
# Evaluate on the untouched test split: AUC / P / R / threshold calibration
from sklearn.metrics import average_precision_score, precision_recall_curve

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_score = np.concatenate([m.predict(x, verbose=0)[:, 0] for x, _ in test_ds])
from sklearn.metrics import roc_auc_score
auc = float(roc_auc_score(y_true, y_score))
ap = float(average_precision_score(y_true, y_score))
prec, rec, thr = precision_recall_curve(y_true, y_score)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
best = float(thr[np.argmax(f1[:-1])])
print(f'test n={len(y_true)} positive rate={y_true.mean():.3f}')
print(f'test AUC={auc:.3f} AP={ap:.3f} | F1-best threshold={best:.3f}')
print('confusion @', round(best, 2), '->',
      {'TP': int(((y_score > best) & (y_true == 1)).sum()),
       'FP': int(((y_score > best) & (y_true == 0)).sum()),
       'TN': int(((y_score <= best) & (y_true == 0)).sum()),
       'FN': int(((y_score <= best) & (y_true == 1)).sum())})
# Overfit gauge: AUC on an un-augmented train sample vs the held-out test set.
train_gauge = tf.keras.utils.image_dataset_from_directory(
    root / 'train', image_size=(IMG, IMG), batch_size=32, shuffle=True).take({'smoke': 3, 'demo': 63, 'full': 128}[SCALE])
train_gauge = train_gauge.map(lambda x, y: (x / 255.0, y)).prefetch(PREFETCH)
y_g = np.concatenate([y.numpy() for _, y in train_gauge])
y_g_score = np.concatenate([m.predict(x, verbose=0)[:, 0] for x, _ in train_gauge])
train_auc = float(roc_auc_score(y_g, y_g_score))
print(f'overfit gauge: train AUC (2000-img sample)={train_auc:.3f} | test AUC={auc:.3f} | gap={train_auc - auc:+.3f}')


2026-08-05 09:24:16.881307: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


test n=5547 positive rate=0.396
test AUC=0.504 AP=0.402 | F1-best threshold=0.167
confusion @ 0.17 -> {'TP': 2193, 'FP': 3342, 'TN': 9, 'FN': 3}
Found 16638 files belonging to 2 classes.


2026-08-05 09:25:32.634440: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


overfit gauge: train AUC (2000-img sample)=0.493 | test AUC=0.504 | gap=-0.012


### Export contract (consumed by the AI service)

The cells below write `../models/nsfw_classifier.onnx` and its dynamic-INT8 quantized copy
`nsfw_classifier_int8.onnx`. `app/ml/serving.py::load_preferred('nsfw_classifier')` loads the
`_int8.onnx` artifact from `AI_MODEL_CACHE_DIR` (dev: bind-mounted to
`backend/ai_service/models/`). The model card JSON is what the `apps.ai` Django
`ModelMetadata` sync endpoint expects.


In [7]:
# Export ONNX (+ INT8) for app/ml/serving.py::load_preferred('nsfw_classifier')
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log

onnx = export_keras_onnx(m, Path('../models'), 'nsfw_classifier', '1.0.0')
q = quantize_dynamic_onnx(onnx)
mlflow_log({'name': 'nsfw_classifier', 'version': '1.0.0',
            'artifact_path': str(q), 'framework': 'tensorflow',
            'metrics': {'test_auc': float(auc), 'test_ap': float(ap), 'threshold': float(best)}})
print('exported', q)

I0000 00:00:1785911160.666708 1062555 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1785911160.666869 1062555 single_machine.cc:376] Starting new session
I0000 00:00:1785911161.675028 1062555 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1785911161.675251 1062555 single_machine.cc:376] Starting new session


{
  "name": "nsfw_classifier",
  "version": "1.0.0",
  "artifact_path": "../models/nsfw_classifier-1.0.0_int8.onnx",
  "framework": "tensorflow",
  "metrics": {
    "test_auc": 0.504421171615574,
    "test_ap": 0.4022916539517874,
    "threshold": 0.1666230410337448
  }
}
exported ../models/nsfw_classifier-1.0.0_int8.onnx
